In [1]:
!wget https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip
!unzip -q maestro-v3.0.0-midi.zip
!pip install pretty_midi

--2026-10-14 07:15:29--  https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 192.178.212.207, 172.217.214.207, 108.177.121.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|192.178.212.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 58416533 (56M) [application/octet-stream]
Saving to: ‘maestro-v3.0.0-midi.zip.4’

maestro-v3.0.0-midi 100%[===================>]  55.71M   148MB/s    in 0.4s    

2026-10-14 07:15:29 (148 MB/s) - ‘maestro-v3.0.0-midi.zip.4’ saved [58416533/58416533]

replace maestro-v3.0.0/2004/MIDI-Unprocessed_XP_08_R1_2004_01-02_ORIG_MID--AUDIO_08_R1_2004_01_Track01_wav.midi? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [19]:
import pretty_midi
from pathlib import Path
from tqdm import tqdm

files = sorted(Path("maestro-v3.0.0").rglob("*.midi"))
print(len(files))   # should be around 1,276
print(files[0])

1276
maestro-v3.0.0/2004/MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_05_Track05_wav.midi


In [20]:
"""
CALCULATE MIN AND MAX IF NECESSARY
MAX = 0
MIN = float('inf')

for file in tqdm(files):
    pm = pretty_midi.PrettyMIDI(str(file))
    piano = pm.instruments[0]
    for note in piano.notes:
        if MAX < note.pitch: MAX = note.pitch
        if MIN > note.pitch: MIN = note.pitch"""

100%|██████████| 1276/1276 [05:30<00:00,  3.87it/s]


In [23]:
MAX = 108
MIN = 21

In [32]:
import numpy as np
import torch

SCALE = 0.001
MAXTIME = 100
pieces = []

class Note():
    def __init__(self, pitch, time, timing):
        self.pitch = pitch
        self.time = int(round(time / SCALE))
        self.timing = timing

def make_tokens(file):
    j = k = 0
    notes = []
    pm = pretty_midi.PrettyMIDI(str(file))
    piano = next(inst for inst in pm.instruments if not inst.is_drum)
    for note in piano.notes:
            note_1 = Note(note.pitch, note.start, 'START')
            note_2 = Note(note.pitch, note.end, 'END')
            notes.append(note_1)
            notes.append(note_2)
    sorted_notes = sorted(notes, key=lambda x: x.time)

    final_notes = []
    for i in range(len(sorted_notes) - 1):
        final_notes.append((sorted_notes[i].timing, sorted_notes[i].pitch))
        time_step = sorted_notes[i+1].time - sorted_notes[i].time
        if time_step != 0:
            while time_step > MAXTIME:
                final_notes.append(('TIME', MAXTIME))
                time_step -= MAXTIME
            final_notes.append(('TIME', time_step))

    final_notes.append((sorted_notes[-1].timing, sorted_notes[-1].pitch))
    return final_notes

In [33]:
token_sqs = [make_tokens(file) for file in tqdm(files)]

100%|██████████| 1276/1276 [11:14<00:00,  1.89it/s]


In [47]:
token_sqs_lens

[36266,
 6545,
 11811,
 25443,
 80019,
 24062,
 41352,
 16730,
 30252,
 24501,
 32535,
 36390,
 23019,
 9080,
 12825,
 41549,
 35441,
 55306,
 14748,
 7851,
 7842,
 12300,
 8424,
 9056,
 13375,
 8290,
 12543,
 52382,
 54252,
 10190,
 25031,
 22382,
 11382,
 14659,
 26745,
 22330,
 15962,
 7703,
 22709,
 64587,
 37910,
 25274,
 6611,
 34542,
 54430,
 13332,
 25068,
 84821,
 22369,
 26568,
 13609,
 76050,
 76145,
 24084,
 8388,
 15553,
 7932,
 32943,
 29107,
 53559,
 87617,
 49051,
 29188,
 14514,
 37709,
 28772,
 59860,
 26546,
 13254,
 12094,
 11449,
 8381,
 12069,
 20922,
 21172,
 6217,
 30293,
 5084,
 54602,
 26914,
 70251,
 26215,
 51115,
 15869,
 31985,
 10587,
 46887,
 10956,
 5279,
 5271,
 35081,
 13283,
 13027,
 23525,
 25751,
 78702,
 45622,
 54137,
 23410,
 24271,
 18588,
 23682,
 35188,
 13123,
 13156,
 23854,
 26237,
 15114,
 11497,
 26588,
 13068,
 10910,
 8335,
 18040,
 57953,
 8239,
 7959,
 78639,
 21188,
 13932,
 37547,
 26710,
 14606,
 38756,
 30089,
 17864,
 27724,
 73

In [46]:
token_sqs_lens = [len(token_sq) for token_sq in token_sqs]
print(f"MAX: {max(token_sqs_lens*1)} MIN: {min(token_sqs_lens*1)} MEAN: {sum(token_sqs_lens)/len(token_sqs_lens)}")

TypeError: 'int' object is not callable

In [55]:
def make_vocab():
    vocab = {'PAD': 0, 'BOS': 1, 'EOS': 2}
    for kind in ('START', 'END'):
        for p in range(MIN, MAX+1):
            vocab[(kind, p)] = len(vocab)
    for t in range(1, MAXTIME+1):
        vocab[('TIME', t)] = len(vocab)
    return vocab

def make_ids(tokens, vocab):
    ids = [vocab['BOS']]
    for key, val in tokens:
        ids.append(vocab[(key, val)])
    ids.append(vocab['EOS'])
    return ids

In [71]:
import torch

class MidiDataset(torch.utils.data.Dataset):
    def __init__(self, tokens_sqs, vocab, sq_len=100):
        self.sqs = [torch.tensor(make_ids(sq, vocab), dtype=torch.long) for sq in tqdm(tokens_sqs)]
        self.sq_len = sq_len

    def __len__(self):
        return len(self.sqs)

    def __getitem__(self, i):
        x = self.sqs[i][:self.sq_len]
        return torch.nn.functional.pad(x, (0, self.sq_len - len(x)), value=0)


In [155]:
vocab = make_vocab()
inv_vocab = {v: k for k,v in vocab.items()}
vocab_size = len(vocab)

In [73]:
data = MidiDataset(token_sqs, vocab)

100%|██████████| 1276/1276 [00:10<00:00, 116.50it/s]


In [83]:
from torch.utils.data import DataLoader
def make_xy(x):
    return x[..., :-1], x[..., 1:]

dataloader = DataLoader(data, batch_size=16, shuffle=True)
batch = next(iter(dataloader))            # tensor (16, seq_len + 1)
x, y = make_xy(batch)              # each (16, seq_len)"""

tensor([[ 60, 183,  48,  ..., 210, 146, 199],
        [ 61, 184,  13,  ..., 278, 278, 278],
        [ 24, 278, 278,  ...,  24, 278, 278],
        ...,
        [ 21, 182,  22,  ..., 110, 268,  29],
        [ 27, 197,  42,  ..., 181, 147, 183],
        [ 41, 232,  40,  ..., 217, 110, 278]])

In [87]:
def make_xy(x):
    return x[:-1], x[1:]

x, y = make_xy(data[0])

print("----------------------------")
print("input:", x)
print("target: ", y)
print("----------------------------")

----------------------------
input: tensor([  1,  53, 275, 141, 267,  37, 188,  53, 278, 253,  41, 211, 125, 278,
        212, 129, 180,  44, 278, 198, 132, 211,  54, 186, 141, 187,  49, 203,
        142, 278, 233,  56,  39, 195, 137, 216,  54, 238, 144,  56, 186, 142,
        221,  54, 201,  49, 188, 144, 276, 137, 226,  48, 278, 259, 136, 189,
        142, 181, 127,  53, 213,  46, 210, 141, 179,  54, 261, 142, 183,  56,
        211,  41, 230, 134, 278, 202,  44, 278, 270,  48, 212, 132, 278, 214,
         49, 201, 136, 278, 225, 129, 206, 144, 183,  53, 278, 227, 141, 196,
         56])
target:  tensor([ 53, 275, 141, 267,  37, 188,  53, 278, 253,  41, 211, 125, 278, 212,
        129, 180,  44, 278, 198, 132, 211,  54, 186, 141, 187,  49, 203, 142,
        278, 233,  56,  39, 195, 137, 216,  54, 238, 144,  56, 186, 142, 221,
         54, 201,  49, 188, 144, 276, 137, 226,  48, 278, 259, 136, 189, 142,
        181, 127,  53, 213,  46, 210, 141, 179,  54, 261, 142, 183,  56, 211,
     

In [94]:
import torch.nn as nn
torch.manual_seed(1502)

class BigramModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx):
        logits = self.token_embedding_table(idx)
        return logits

def generate(bigrammodel, max_tokens, idx=1): #idx=1=BOS
    sq = [1]
    for _ in range(max_tokens):
        logits = m(torch.tensor(idx, dtype=torch.long))
        probs = nn.functional.softmax(logits, dim=-1)
        idx = torch.multinomial(probs, num_samples = 1).item()
        sq.append(idx)
    return sq


m = BigramModel(vocab_size)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=m.parameters(), lr=1e-3)

In [107]:
for epoch in range(1000):
    x, y = make_xy(data[epoch])
    y_pred = m(x)
    loss = loss_fn(y_pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
print(loss.item())

3.8518877029418945


In [157]:
s = generate(m, 100)
sa = [inv_vocab[a] for a in s]
sa

['BOS',
 ('START', 60),
 ('TIME', 19),
 ('START', 80),
 ('TIME', 96),
 ('END', 70),
 ('TIME', 100),
 ('TIME', 100),
 ('TIME', 100),
 ('TIME', 56),
 ('END', 58),
 ('END', 63),
 ('TIME', 88),
 ('START', 78),
 ('TIME', 4),
 ('START', 76),
 ('TIME', 92),
 ('START', 49),
 ('TIME', 11),
 ('START', 42),
 ('TIME', 31),
 ('END', 37),
 ('TIME', 45),
 ('END', 71),
 ('TIME', 29),
 ('END', 67),
 ('TIME', 10),
 ('START', 55),
 ('TIME', 64),
 ('END', 51),
 ('TIME', 14),
 ('START', 75),
 ('TIME', 56),
 ('END', 73),
 ('TIME', 36),
 ('END', 74),
 ('TIME', 16),
 ('START', 80),
 ('TIME', 13),
 ('END', 46),
 ('TIME', 6),
 ('END', 57),
 ('TIME', 100),
 ('TIME', 82),
 ('END', 47),
 ('TIME', 60),
 ('START', 47),
 ('TIME', 4),
 ('START', 43),
 ('TIME', 35),
 ('END', 48),
 ('TIME', 100),
 ('TIME', 100),
 ('TIME', 100),
 ('TIME', 100),
 ('TIME', 100),
 ('TIME', 100),
 ('TIME', 100),
 ('TIME', 100),
 ('TIME', 100),
 ('TIME', 27),
 ('START', 77),
 ('TIME', 25),
 ('START', 67),
 ('TIME', 100),
 ('TIME', 100),
 ('TI

In [148]:
logits = m(torch.tensor(1, dtype=torch.long))
probs = nn.functional.softmax(logits)
torch.multinomial(probs, num_samples=1).item()

/tmp/ipykernel_19070/2638874780.py:2: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  probs = nn.functional.softmax(logits)


49

In [ ]:
 print((data != 0).sum().item() / len(data) * 100)

20.227097310455385


In [ ]:
import numpy as np
import pretty_midi
from collections import defaultdict, deque
from IPython.display import Audio, display
import fluidsynth

def stream_to_midi(piece, scale=0.01, velocity=80, program=0):
    piece = np.asarray(piece)              # works for torch tensors on CPU too
    pm = pretty_midi.PrettyMIDI()
    inst = pretty_midi.Instrument(program=program)   # 0 = acoustic grand piano

    open_notes = defaultdict(deque)        # pitch -> start times waiting for an off
    for idx in np.flatnonzero(piece):
        v = int(round(piece[idx]))
        t = idx * scale
        if v > 0:
            open_notes[v].append(t)
        else:
            pitch = -v
            if open_notes[pitch]:
                start = open_notes[pitch].popleft()   # FIFO pairing
                inst.notes.append(pretty_midi.Note(velocity, pitch, start, t))

    pm.instruments.append(inst)
    return pm

pm_out = stream_to_midi(data[1].numpy() if hasattr(piece, "numpy") else piece, scale=0.01)

# Listen in the notebook
sf2 = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
audio = pm_out.fluidsynth(fs=44100, sf2_path=sf2)
display(Audio(audio, rate=44100))

/tmp/ipykernel_24606/3489307696.py:31: DeprecationWarning: The parameter 'sf2_path' is deprecated, please use 'synthesizer' instead.
  audio = pm_out.fluidsynth(fs=44100, sf2_path=sf2)


ValueError: zero-size array to reduction operation maximum which has no identity

In [ ]:
#ORIGINAL AUDIO
audio = pm.fluidsynth(fs=44100, sf2_path=sf2)
display(Audio(audio, rate=44100))

In [ ]:
n = int(0.9*len(data))
data_train = data[:n]
data_test = data[n:]

In [ ]:
#manual seef
torch.manual_seed(1502)

In [ ]:
BATCH_SIZE = 4
BLOCK_SIZE = 50

def get_batch(data, batch_size, block_size):
    ix = torch.randint(len(data)-block_size, (batch_size, ))
    print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch(data_train, BATCH_SIZE, BLOCK_SIZE)

In [ ]:
vocab_size = len(data_train.unique())

In [ ]:
print(xb)

tensor([[  0,   0,   0,   0, -72,   0,  63,  60,   0,   0,   0,   0,   0,   0,
         -63, -60,  68,  51,   0,   0,   0,   0, -68,  72,   0, -51,   0,   0,
           0,   0,   0,   0, -72,  44,  75,   0,   0,   0, -44,   0,   0,   0,
          51,   0,   0,  68,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,  83,   0,   0,   0,   0,   0,   0, -83,   0,   0,
           0,  92,   0,   0,   0,  83,   0,   0,   0,   0,   0,   0,   0, -83,
          56,  88, -92, -52,   0,   0,   0,   0, -88,   0,   0,   0,  80,   0,
           0,   0,   0, -80,   0,   0,   0,   0],
        [ 57,  79, -74, -78, -57,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,  78, -79,   0,   0,   0,   0,   0,  

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLangModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]               # (B, C)
            probs = F.softmax(logits, dim=-1)       # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLangModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)   # (B*T, vocab_size)
print(loss)

IndexError: index out of range in self